# Lab 1.3 &mdash; create_agent, and What a Tool Description Is Worth

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Build the same agent twice &mdash; once with opaque tool descriptions, once with good ones
- <b>Measure</b> the difference in tool-selection accuracy against the live model
- Make the agent return a typed <code>Verdict</code> with <code>response_format</code>, not prose
- Read the message trace `create_agent` produces, and find where a run went wrong

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Labs 1.1 and 1.2.** You know what the loop does; now you use the built one
> and spend your effort on the part that actually decides whether it works.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

`create_agent(model, tools, system_prompt)` is the loop from Lab 1.1, hardened. Which means the
interesting engineering moves somewhere else &mdash; to the three things you still control:

1. **the tool descriptions**, which are the only guide the model has when choosing,
2. **the system prompt**, which sets the procedure,
3. **the output contract**, which decides whether the caller gets prose or data.

This lab measures the first and fixes the third. Everything here is a real agent making real
calls, so the numbers you get are yours, not slides.

## Section 1 &mdash; The same five tools, described two ways

Both toolsets do **exactly the same work**. Only the names and descriptions differ. That is the
experiment: nothing changes except what the model can read.

Write the four missing descriptions in `GOOD`. A good one says what the tool returns, when to
reach for it, and when not to.

In [ ]:
from langchain_core.tools import tool, StructuredTool

# The five underlying operations, as plain functions. Shared by both arms.
def _payment(ref: str) -> str:
    r = LEDGER.get(ref)
    return json.dumps({"ref": ref, **r}) if r else f"no payment found with reference {ref!r}"

def _policy(reason_code: str) -> str:
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")

def _counterparty(name: str) -> str:
    hits = [k for k, v in LEDGER.items() if v["counterparty"] == name]
    return json.dumps({"counterparty": name, "payments": hits}) if hits else f"no counterparty {name!r}"

def _by_status(status: str) -> str:
    hits = [k for k, v in LEDGER.items() if v["status"] == status]
    return json.dumps({"status": status, "payments": hits})

def _needs_human(reason_code: str) -> str:
    return json.dumps({"reason_code": reason_code, "needs_human": reason_code in NEEDS_HUMAN})

OPS = {"payment": _payment, "policy": _policy, "counterparty": _counterparty,
       "by_status": _by_status, "needs_human": _needs_human}

# --- arm A: opaque. A name and a shrug -- what a rushed codebase actually looks like.
POOR = [
    StructuredTool.from_function(_payment,      name="tool_a", description="Gets data."),
    StructuredTool.from_function(_policy,       name="tool_b", description="Gets data."),
    StructuredTool.from_function(_counterparty, name="tool_c", description="Looks things up."),
    StructuredTool.from_function(_by_status,    name="tool_d", description="Looks things up."),
    StructuredTool.from_function(_needs_human,  name="tool_e", description="Checks something."),
]

# --- arm B: described. Same functions, same order.
GOOD = [
    StructuredTool.from_function(
        _payment, name="lookup_payment",
        description="Return the full ledger record (amount, currency, counterparty, status, "
                    "reason code) for ONE payment reference such as 'PMT-1003'. Use when you "
                    "have a reference. Not for searching."),
    StructuredTool.from_function(
        _policy, name="policy_for",
        description="Return the operating policy text for ONE failure reason code such as "
                    "'LIMIT_BREACH' or 'SANCTIONS_REVIEW'. Use after you know why a payment "
                    "failed and need to know what to do about it. Not for looking up payments."),
    StructuredTool.from_function(
        _counterparty, name="payments_for_counterparty",
        description="Return every payment reference belonging to ONE counterparty name such as "
                    "'NORTHWIND'. Use when the question names a party rather than a reference. "
                    "Returns references only -- call lookup_payment for the details of each."),
    StructuredTool.from_function(
        _by_status, name="payments_by_status",
        description="Return every payment reference with a given status. Valid statuses are "
                    "exactly 'settled', 'failed' and 'held'. Use for questions about a whole "
                    "queue, such as what is currently held."),
    StructuredTool.from_function(
        _needs_human, name="requires_human_approval",
        description="Return whether a reason code obliges a human decision before any action. "
                    "Call it before proposing to release, cancel or repair a payment; if it "
                    "returns true you must stop and escalate rather than act."),
]

In [ ]:
# --- Self-check: Section 1   (tool objects only -- no model call)
def _descs():
    out = []
    for t in GOOD:
        d = (t.description or "").strip()
        if d == "BLANK" or not d:
            raise NameError(f"{t.name} still has no description")
        out.append(d)
    return out

check("both arms expose five tools", lambda: len(POOR) == 5 and len(GOOD) == 5)
check("the two arms wrap the same functions",
      lambda: [t.func for t in POOR] == [t.func for t in GOOD],
      "the ONLY difference between the arms must be name and description")
check("every GOOD tool has a real description",
      lambda: all(len(d) > 60 for d in _descs()),
      "a one-liner is not a description -- say what it returns and when to use it")
check("the descriptions distinguish the two lookup tools",
      lambda: "not for searching" in _descs()[0].lower()
              and any("reference" in d.lower() for d in _descs()[2:3]))
check("payments_by_status names its valid values",
      lambda: all(s in _descs()[3] for s in ("settled", "failed", "held")),
      "the model cannot guess an enum it was never shown")
check("the POOR arm really is uninformative",
      lambda: all(len(t.description) < 25 for t in POOR))

## Section 2 &mdash; The harness that scores a run

For each question we know which tool *should* be called first. `first_tool()` digs that out of
the trace `create_agent` returns; `bake_off()` runs a whole set and reports a pass rate.

This is your first eval harness. Day 2 measures a multi-agent graph against exactly this shape.

In [ ]:
CASES = [
    # question,                                                    poor name,  good name
    ("What is the status of PMT-1003?",                            "tool_a",  "lookup_payment"),
    ("What should we do about a LIMIT_BREACH?",                    "tool_b",  "policy_for"),
    ("Which payments belong to NORTHWIND?",                        "tool_c",  "payments_for_counterparty"),
    ("List everything currently held.",                            "tool_d",  "payments_by_status"),
    ("Does a SANCTIONS_REVIEW need a person to sign it off?",      "tool_e",  "requires_human_approval"),
]

def first_tool(result: dict) -> str | None:
    """The name of the FIRST tool the agent chose, from the messages it returned."""
    for m in result["messages"]:
        calls = getattr(m, "tool_calls", None)
        if calls:
            return calls[0]["name"]
    return None


def bake_off(agent, arm: str) -> dict:
    """Run every case through `agent`; score the first tool chosen against the expectation."""
    idx = 1 if arm == "poor" else 2
    hits, rows = 0, []
    for case in CASES:
        question, expected = case[0], case[idx]
        try:
            chosen = first_tool(agent.invoke({"messages": [("human", question)]}))
        except Exception as exc:
            chosen = f"<error: {type(exc).__name__}>"
        ok = chosen == expected
        hits += ok
        rows.append((question, expected, chosen, ok))
    return {"arm": arm, "hits": hits, "of": len(CASES),
            "rate": hits / len(CASES), "rows": rows}

In [ ]:
# --- Self-check: Section 2   (canned traces -- no model call)
from langchain_core.messages import AIMessage, HumanMessage

_canned = {"messages": [
    HumanMessage("q"),
    AIMessage(content="", tool_calls=[{"name": "lookup_payment", "args": {"ref": "PMT-1003"},
                                       "id": "c1", "type": "tool_call"}]),
    AIMessage("done"),
]}
_no_tools = {"messages": [HumanMessage("q"), AIMessage("answered from memory")]}

check("first_tool finds the first chosen tool",
      lambda: first_tool(_canned) == "lookup_payment")
check("first_tool returns None when no tool was used",
      lambda: first_tool(_no_tools) is None,
      "an agent that answers without a tool is a result, not a crash")
check("every case names a tool that exists in both arms",
      lambda: all(c[1] in {t.name for t in POOR} and c[2] in {t.name for t in GOOD} for c in CASES))
check("the cases cover all five tools",
      lambda: len({c[2] for c in CASES}) == 5)

## Section 3 &mdash; A typed answer, not a paragraph

An agent that returns prose forces every caller to parse it. `response_format=Verdict` makes
`create_agent` return a validated `Verdict` object alongside the messages, in
`result["structured_response"]`.

Declare the contract you would want if you had to call this service from another system.

In [ ]:
from pydantic import BaseModel, Field

class Verdict(BaseModel):
    """The outcome of investigating one payment exception."""
    ref: str = Field(description="The payment reference investigated, e.g. 'PMT-1003'")
    reason_code: str = Field(description="The ledger reason code, or 'NONE' if the payment is fine")
    needs_human: bool = Field(
        description="True if policy requires a named human to decide before any action is taken; "
                    "false only if the agent may act on its own authority")
    action: str = Field(description="The single next action, in one short line")
    evidence: str = Field(description="The policy text or ledger field that justifies the action")

In [ ]:
# --- Self-check: Section 3   (schema only -- no model call)
def _needs_human_desc():
    d = Verdict.model_fields["needs_human"].description
    if not d or d == "BLANK":
        raise NameError("needs_human still has no description")
    return d

check("the contract has all five fields",
      lambda: set(Verdict.model_fields) == {"ref", "reason_code", "needs_human", "action", "evidence"})
check("every field carries a description",
      lambda: all(f.description for f in Verdict.model_fields.values()),
      "with_structured_output sends these to the model -- an undescribed field is a guess")
check("needs_human says what true MEANS",
      lambda: len(_needs_human_desc()) > 40 and "human" in _needs_human_desc().lower())
check("a Verdict validates",
      lambda: Verdict(ref="PMT-1003", reason_code="LIMIT_BREACH", needs_human=True,
                      action="Escalate to Treasury", evidence="above USD 500,000").needs_human is True)

## Run it for real &mdash; the description experiment

Two agents. Same model, same functions, same questions. Only the descriptions differ.

Ten agent runs, so give it a moment.

In [ ]:
if llm_ready():
    from langchain.agents import create_agent

    def _experiment():
        sysmsg = ("You are a payments operations analyst. Use exactly one tool to answer, "
                  "then reply. Do not guess if a tool can tell you.")
        results = {}
        for arm, tools in (("poor", POOR), ("good", GOOD)):
            ag = create_agent(model=get_llm(), tools=tools, system_prompt=sysmsg)
            r = bake_off(ag, arm)
            results[arm] = r
            print(f"\n=== {arm.upper()} descriptions: {r['hits']}/{r['of']} correct "
                  f"({r['rate']:.0%}) ===")
            for q, expected, chosen, ok in r["rows"]:
                print(f"  [{'ok ' if ok else 'MISS'}] {q[:46]:48} want={expected:26} got={chosen}")
        d = results["good"]["rate"] - results["poor"]["rate"]
        print(f"\nDelta from description quality alone: {d:+.0%}")
        return results
    RESULTS = guard(_experiment)

### Read it

Nothing about the model, the questions or the underlying functions changed between those two
runs. Whatever gap you just measured is the value of writing a sentence.

Two things worth noticing in the misses. First, where the opaque arm guessed, it usually guessed
the *first* tool &mdash; with nothing to choose on, order becomes the tiebreak. Second, a miss is not
always a wrong answer: the agent sometimes recovers by calling a second tool, which costs tokens
and latency rather than correctness. That distinction is the whole subject of Module 7.

## Run it for real &mdash; the typed answer

Now the same agent with a contract. Note that the caller never touches `.content`.

In [ ]:
if llm_ready():
    def _typed():
        ag = create_agent(
            model=get_llm(), tools=GOOD,
            system_prompt=("You investigate payment exceptions. Procedure, in order: "
                           "1) look up the payment; 2) look up the policy for its reason code; "
                           "3) check whether it requires human approval; 4) answer."),
            response_format=Verdict)
        out = ag.invoke({"messages": [("human", "Investigate PMT-1005 and tell me what to do.")]})
        v = out.get("structured_response")
        if v is None:
            # This happens, and it is worth seeing rather than hiding. response_format asks
            # the model to finish by calling a Verdict tool; if it answers in prose instead,
            # there is no object -- and a caller expecting one gets None, not an error.
            print("NO structured_response -- the model answered without filling the contract.")
            print("Look at the last message: it replied in prose instead of calling Verdict.\n")
        else:
            print(f"ref         : {v.ref}")
            print(f"reason_code : {v.reason_code}")
            print(f"needs_human : {v.needs_human}")
            print(f"action      : {v.action}")
            print(f"evidence    : {v.evidence}")
        print(f"\n--- the trace behind it ({len(out['messages'])} messages) ---")
        show_messages(out["messages"])
        return v
    VERDICT = guard(_typed)

### Read the trace

`needs_human` should be **true** for PMT-1005 &mdash; it is a sanctions hold, and the policy says
Compliance decides. If it came back false, the trace tells you which step was skipped: look for
whether `requires_human_approval` was ever called at all.

And you may have got **no object at all**. `response_format` asks the model to finish by calling
a `Verdict` tool; a model that decides to reply in prose instead leaves
`result["structured_response"]` as `None`. Nothing raises. Run the cell a few times &mdash; on this
model it does not happen every time, which is worse than if it never worked.

That is the honest lesson of this lab, in two parts. A typed contract guarantees the *shape* of
the answer **when you get one**, so a caller must still handle its absence. And it guarantees
nothing at all about the truth of it &mdash; the system prompt's ordered procedure is doing that
work, and Module 8 is where you learn not to trust either without a check.

In [ ]:
score()

## Your turn

1. Build a third arm, `MEDIUM` &mdash; real names, but one-line descriptions with no "not for"
   clause. Where does it land between the two? That gap is the value of the negative half.
2. Add a sixth tool that genuinely overlaps with an existing one (`get_status(ref)` beside
   `lookup_payment`) and re-run. Which description do you have to change to fix the confusion?
3. Remove the numbered procedure from the system prompt in the typed run and re-run it five
   times. Count how often `needs_human` comes back wrong. That number is your argument for
   Module 8's guardrails.